# AI watermark solutions

Research companion for [issue #115](https://github.com/pomodorozhong/personal-research/issues/115). The notebook compares provenance credentials and signal watermarks, then builds small image and text demonstrations that make the core implementation trade-offs visible.

The demos are deliberately toy implementations. They explain the pattern of embedding, transforming and detecting a signal; they do not reproduce the security, capacity or robustness of SynthID, AudioSeal, VideoSeal or a production C2PA implementation.

## Executive summary

The most useful distinction is between **provenance** and **watermarking**:

- C2PA Content Credentials use signed manifests to state who created or edited an asset and what happened to it.
- A robust invisible watermark changes pixels, samples or token choices so a detector can still find a provider-specific signal after common transformations.
- The two layers answer different questions and work best together. A valid signature is evidence about a signed claim; a detected watermark is evidence that a detector recognized a signal. Neither one proves that an unmarked asset is human-made.

Representative solutions:

| Family | Medium | Mechanism | Practical role |
| --- | --- | --- | --- |
| [C2PA](https://spec.c2pa.org/specifications/specifications/2.4/specs/ContentCredentials.html) | Image, video, audio, text and more | Signed manifest with hard and soft bindings | Provenance and edit history |
| [SynthID](https://deepmind.google/models/synthid/) | Image, video, audio, text | Provider-specific invisible signal; token-probability modulation for text | Attribution inside Google products |
| [Stable Signature](https://arxiv.org/abs/2303.15435) | Image | Root a signal in a diffusion decoder | Watermark images at generation time |
| [AudioSeal](https://github.com/facebookresearch/audioseal) | Speech/audio | Joint generator and localized detector | Voice-clone and generated-speech detection |
| [VideoSeal](https://github.com/facebookresearch/videoseal) | Image/video | Neural encoder/extractor with temporal consistency | Open post-hoc media watermarking |
| LLM green-list methods | Text | Keyed token partition and sampling bias | Statistical detection of generated text |


## 1. Choose the layer from the threat model

Before choosing an algorithm, define what must survive and what the detector is allowed to know.

| Goal | Prefer | Why |
| --- | --- | --- |
| Show origin, signer and edit history | C2PA manifest and signature | The claim is explicit and tamper-evident |
| Recover provenance after metadata is stripped | C2PA soft binding plus a manifest store | A fingerprint or watermark can find the detached manifest |
| Attribute output to a controlled image/video model | In-model or post-hoc neural watermark | The model can embed a secret-keyed signal at generation time |
| Detect generated speech inside a long recording | Localized audio watermark | The detector can report which samples are watermarked |
| Detect text from a known sampler | Generation-time statistical watermark | It can be inserted without changing the rendered text format |
| Prove that an asset is not AI-generated | None of the above alone | Absence of a signal is not proof of human authorship |


### C2PA is a provenance layer, not a magic watermark

C2PA manifests are signed claims about an asset. A hard binding hashes the asset or file structures. A soft binding can use a fingerprint or invisible watermark to help locate a manifest after ordinary transformations or when the embedded manifest is removed. The signature, key custody and validator establish trust in the claim; the soft binding only helps match the rendition back to that claim.

A production pipeline therefore looks like:

```text
create/edit asset
    -> write a signed C2PA manifest
    -> embed a robust media watermark or fingerprint
    -> store the manifest and detector metadata
publish rendition
    -> validate signature and hard binding
    -> if missing, resolve by soft binding
```

Do not put a private key or a detector secret in a client-side notebook. In a service, keys belong in a KMS/HSM and detector results should be logged with model/version, threshold and calibration-set identifiers.

In [ ]:
import io
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

try:
    import ipywidgets as widgets
    HAS_WIDGETS = True
except ImportError:
    widgets = None
    HAS_WIDGETS = False

RNG = np.random.default_rng(115)
RUN_INTERACTIVE = HAS_WIDGETS and os.environ.get('WATERMARKS_INTERACTIVE', '1') != '0'
print(f'ipywidgets available: {HAS_WIDGETS}')


## 2. Interactive image example: a small DCT watermark

A classical image watermark can modify mid-frequency coefficients in 8x8 luminance blocks. Low frequencies are visible when changed; high frequencies are easily removed by resampling and JPEG quantization. This demo encodes each payload bit by forcing the sign of the difference between two mid-frequency coefficients. The detector reads every block and uses majority vote across repeated copies of the payload.

This is a useful mental model for the robustness/imperceptibility trade-off:

- `strength` increases the separation between the two coefficients, improving detection margin but increasing pixel error.
- Repetition gives redundancy but consumes capacity.
- JPEG, noise and resize are ordinary transformations; an adversary with access to the detector can try targeted removal or forging attacks.


In [ ]:
def make_test_image(size=256):
    yy, xx = np.mgrid[0:size, 0:size].astype(float)
    x = xx / size
    y = yy / size
    texture = np.sin(xx / 5.5) * np.cos(yy / 7.0)
    red = 55 + 165 * x + 8 * texture
    green = 70 + 135 * y + 5 * texture
    blue = 180 - 85 * x + 10 * np.sin(yy / 11.0)
    return np.clip(np.stack([red, green, blue], axis=-1), 0, 255).astype(np.uint8)

def dct_matrix(n=8):
    i, k = np.arange(n)[:, None], np.arange(n)[None, :]
    basis = np.cos(np.pi / n * (i + 0.5) * k)
    basis[:, 0] /= np.sqrt(2)
    return np.sqrt(2 / n) * basis

DCT8 = dct_matrix()

def payload_bits(text):
    raw = np.frombuffer(text.encode('utf-8'), dtype=np.uint8)
    return np.unpackbits(raw).astype(np.uint8)

def block_positions(height, width, block_size=8):
    return [(row, col)
            for row in range(0, height - block_size + 1, block_size)
            for col in range(0, width - block_size + 1, block_size)]

def embed_dct(image, bits, strength=12.0):
    if image.shape[0] % 8 or image.shape[1] % 8:
        raise ValueError('image height and width must be divisible by 8')
    rgb = image.astype(float)
    luminance = 0.299 * rgb[..., 0] + 0.587 * rgb[..., 1] + 0.114 * rgb[..., 2]
    modified = luminance.copy()
    positions = block_positions(*luminance.shape)
    for index, (row, col) in enumerate(positions):
        block = luminance[row:row + 8, col:col + 8]
        coeff = DCT8 @ block @ DCT8.T
        difference = coeff[2, 3] - coeff[3, 2]
        desired = strength if bits[index % len(bits)] else -strength
        if bits[index % len(bits)] and difference < desired:
            adjustment = (desired - difference) / 2
            coeff[2, 3] += adjustment
            coeff[3, 2] -= adjustment
        elif not bits[index % len(bits)] and difference > desired:
            adjustment = (difference - desired) / 2
            coeff[2, 3] -= adjustment
            coeff[3, 2] += adjustment
        modified[row:row + 8, col:col + 8] = DCT8.T @ coeff @ DCT8
    residual = modified - luminance
    return np.clip(rgb + residual[..., None], 0, 255).astype(np.uint8)

def detect_dct(image, payload_length):
    rgb = image.astype(float)
    luminance = 0.299 * rgb[..., 0] + 0.587 * rgb[..., 1] + 0.114 * rgb[..., 2]
    positions = block_positions(*luminance.shape)
    signs = []
    margins = []
    for row, col in positions:
        block = luminance[row:row + 8, col:col + 8]
        coeff = DCT8 @ block @ DCT8.T
        difference = coeff[2, 3] - coeff[3, 2]
        signs.append(difference >= 0)
        margins.append(abs(difference))
    signs = np.asarray(signs, dtype=bool)
    decoded = np.asarray([np.mean(signs[index::payload_length]) >= 0.5
                         for index in range(payload_length)], dtype=np.uint8)
    return decoded, np.asarray(margins)

def jpeg_roundtrip(image, quality=55):
    buffer = io.BytesIO()
    Image.fromarray(image).save(buffer, format='JPEG', quality=int(quality))
    return np.asarray(Image.open(io.BytesIO(buffer.getvalue())).convert('RGB'))

def center_crop_resize(image, fraction=0.75):
    height, width = image.shape[:2]
    crop_height, crop_width = int(height * fraction), int(width * fraction)
    top = (height - crop_height) // 2
    left = (width - crop_width) // 2
    cropped = Image.fromarray(image).crop((left, top, left + crop_width, top + crop_height))
    return np.asarray(cropped.resize((width, height), Image.Resampling.LANCZOS))

def bit_accuracy(actual, expected):
    return float(np.mean(np.asarray(actual) == np.asarray(expected)))


In [ ]:
image = make_test_image()
payload = payload_bits('AI115')
watermarked = embed_dct(image, payload, strength=12)
decoded, margins = detect_dct(watermarked, len(payload))

difference = np.abs(watermarked.astype(float) - image.astype(float)).mean(axis=2)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(image)
axes[0].set_title('Original')
axes[1].imshow(watermarked)
axes[1].set_title(f'Watermarked, accuracy={bit_accuracy(decoded, payload):.1%}')
axes[2].imshow(difference, cmap='magma', vmin=0, vmax=max(1, np.percentile(difference, 99)))
axes[2].set_title('Mean absolute pixel change')
for axis in axes:
    axis.axis('off')
plt.tight_layout()
print(f'Payload bits: {len(payload)}; mean detector margin: {margins.mean():.2f}')


In [ ]:
transforms = {
    'none': lambda value: value,
    'JPEG quality 55': lambda value: jpeg_roundtrip(value, quality=55),
    'Gaussian noise': lambda value: np.clip(value.astype(float) + RNG.normal(0, 2.5, value.shape), 0, 255).astype(np.uint8),
    'crop then resize': lambda value: center_crop_resize(value, fraction=0.75),
}

print('{:<20} {:>12} {:>12}'.format('Transform', 'Bit accuracy', 'Mean margin'))
print('-' * 48)
for name, transform in transforms.items():
    transformed = transform(watermarked)
    recovered, recovered_margins = detect_dct(transformed, len(payload))
    print(f'{name:<20} {bit_accuracy(recovered, payload):>11.1%} {recovered_margins.mean():>12.2f}')


The crop result is intentionally informative: a block-based detector that assumes the original geometry loses synchronization when the image is cropped and resized. Research-grade systems address this with synchronization markers, multi-scale features, spatially distributed redundancy, or a learned detector trained on crop/resize augmentations. Robustness is always relative to a specified transformation set.

In [ ]:
def render_image_demo(strength=12.0, jpeg_quality=55):
    current_watermark = embed_dct(image, payload, strength=float(strength))
    compressed = jpeg_roundtrip(current_watermark, quality=int(jpeg_quality))
    decoded_now, _ = detect_dct(compressed, len(payload))
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].imshow(current_watermark)
    axes[0].set_title(f'strength={float(strength):.1f}')
    axes[1].imshow(compressed)
    axes[1].set_title(f'JPEG quality={int(jpeg_quality)}')
    axes[2].bar(np.arange(len(payload)), decoded_now, color='tab:blue')
    axes[2].set_ylim(-0.1, 1.1)
    axes[2].set_title(f'detected={bit_accuracy(decoded_now, payload):.1%}')
    axes[2].set_xlabel('payload bit')
    for axis in axes[:2]:
        axis.axis('off')
    plt.tight_layout()

if RUN_INTERACTIVE:
    widgets.interact(
        render_image_demo,
        strength=widgets.FloatSlider(value=12, min=2, max=30, step=1, description='strength'),
        jpeg_quality=widgets.IntSlider(value=55, min=15, max=95, step=5, description='JPEG'),
    )
else:
    render_image_demo()


## 3. Interactive text example: a keyed green-list watermark

A common language-model watermark partitions the vocabulary using a secret key and the previous context. At each generation step, the sampler slightly increases the probability of tokens in the green list. The detector counts green-list tokens and computes a z-score against the null hypothesis that the text is unwatermarked.

This toy sampler uses random logits instead of a language model, so it is not a quality evaluation. It does show why text watermarks need enough tokens, why detection is statistical, and why paraphrase or translation can erase the signal. In a real system the partition must be keyed to context, the detector must estimate the correct null distribution, and thresholds must be calibrated by model, language and decoding policy.

In [ ]:
VOCABULARY = np.array(
    'the a an useful small large model image video audio text signal hidden visible robust fragile detect detector create edit source human generated content token probability secret key sample transform compression crop noise'.split()
)

def green_mask(position, secret=115, vocabulary_size=len(VOCABULARY)):
    local_rng = np.random.default_rng(int(secret) + 10007 * int(position))
    permutation = local_rng.permutation(vocabulary_size)
    mask = np.zeros(vocabulary_size, dtype=bool)
    mask[permutation[:vocabulary_size // 2]] = True
    return mask

def sample_tokens(length=120, bias=0.0, secret=115, seed=1):
    local_rng = np.random.default_rng(seed)
    tokens = []
    for position in range(length):
        logits = local_rng.normal(0, 1, len(VOCABULARY))
        logits[green_mask(position, secret)] += float(bias)
        probabilities = np.exp(logits - logits.max())
        probabilities /= probabilities.sum()
        tokens.append(local_rng.choice(len(VOCABULARY), p=probabilities))
    return np.asarray(tokens, dtype=int)

def green_rate(tokens, secret=115):
    membership = [green_mask(position, secret)[token] for position, token in enumerate(tokens)]
    return float(np.mean(membership))

def watermark_z_score(tokens, secret=115):
    count = green_rate(tokens, secret) * len(tokens)
    expected = 0.5 * len(tokens)
    standard_deviation = np.sqrt(0.25 * len(tokens))
    return float((count - expected) / standard_deviation)

def render_text(tokens):
    return ' '.join(VOCABULARY[tokens])

clean_tokens = sample_tokens(length=120, bias=0.0, seed=4)
watermarked_tokens = sample_tokens(length=120, bias=1.5, seed=4)
print(f'Unwatermarked z-score: {watermark_z_score(clean_tokens):.2f}')
print(f'Watermarked z-score:   {watermark_z_score(watermarked_tokens):.2f}')
print()
print(render_text(watermarked_tokens)[:500] + ' ...')


In [ ]:
def render_text_demo(length=120, bias=1.5):
    clean = sample_tokens(length=int(length), bias=0.0, seed=9)
    marked = sample_tokens(length=int(length), bias=float(bias), seed=9)
    scores = [watermark_z_score(clean), watermark_z_score(marked)]
    fig, axis = plt.subplots(figsize=(7, 3.5))
    axis.bar(['unwatermarked', 'watermarked'], scores, color=['tab:gray', 'tab:green'])
    axis.axhline(4, color='tab:red', linestyle='--', label='illustrative threshold')
    axis.set_ylabel('green-list z-score')
    axis.set_title(f'{int(length)} tokens, bias={float(bias):.2f}')
    axis.legend()
    plt.tight_layout()

if RUN_INTERACTIVE:
    widgets.interact(
        render_text_demo,
        length=widgets.IntSlider(value=120, min=16, max=300, step=16, description='tokens'),
        bias=widgets.FloatSlider(value=1.5, min=0, max=3, step=0.25, description='bias'),
    )
else:
    render_text_demo()


In [ ]:
def detection_power(lengths=(16, 32, 64, 128, 256), bias=1.5, threshold=4.0, trials=100):
    marked_power = []
    clean_false_positive = []
    for length in lengths:
        marked_scores = []
        clean_scores = []
        for trial in range(trials):
            marked_scores.append(watermark_z_score(sample_tokens(length, bias, seed=1000 + trial)))
            clean_scores.append(watermark_z_score(sample_tokens(length, 0, seed=2000 + trial)))
        marked_power.append(np.mean(np.asarray(marked_scores) >= threshold))
        clean_false_positive.append(np.mean(np.asarray(clean_scores) >= threshold))
    return np.asarray(marked_power), np.asarray(clean_false_positive)

lengths = np.array([16, 32, 64, 128, 256])
power, false_positive = detection_power(lengths)
plt.figure(figsize=(7, 4))
plt.plot(lengths, power, marker='o', label='watermarked detection rate')
plt.plot(lengths, false_positive, marker='o', label='unwatermarked false-positive rate')
plt.xscale('log', base=2)
plt.ylim(-0.02, 1.02)
plt.xlabel('tokens')
plt.ylabel('fraction above z=4')
plt.title('Toy detector power depends on text length')
plt.legend()
plt.tight_layout()


## 4. How the popular systems map onto the demos

### SynthID
Google describes SynthID as a family that embeds imperceptible signals in image, video and audio outputs, and adjusts token probabilities for text. The public descriptions emphasize robustness to common transformations and explicitly note that the system is not a silver bullet. The toy image demo mirrors the high-level idea of a detector reading a distributed signal; the toy text demo mirrors probability modulation and statistical detection.

### Stable Signature
Stable Signature moves image embedding into the generator: the decoder of a latent diffusion model is fine-tuned so generated outputs carry a detector-recognizable signal. This makes the watermark automatic for outputs from that decoder, but it also means the system owner must control or modify the generation stack.

### AudioSeal
AudioSeal jointly trains a generator and detector, uses perceptual masking to keep the residual inaudible, and adds localization so a detector can identify marked fragments inside longer or edited speech. The implementation is open source, but its speech focus and model checkpoint should not be assumed to transfer to music without evaluation.

### VideoSeal
VideoSeal uses a neural embedder/extractor, adversarial or differentiable augmentation during training, and temporal strategies so a video does not need to be treated as unrelated still frames. Important evaluation axes include codec, resize, crop, frame rate, frame insertion/deletion and temporal aggregation.

### C2PA
C2PA is the interoperability layer around claims, signatures, bindings and manifests. It can reference watermark algorithms as soft bindings, but it does not replace the modality-specific embedder/detector.

## 5. Production implementation checklist

1. **Threat model:** list the transformations to survive, the attacker capabilities, and whether the detector is public or private.
2. **Payload:** use a short, authenticated identifier rather than raw user data. Include versioning and a key identifier; protect the payload with an error-correcting code and integrity tag.
3. **Embedding:** choose in-model embedding when you control generation, or a post-hoc encoder when you need model-agnostic coverage. Use perceptual masking so flat or salient regions are not damaged.
4. **Detector:** return a confidence or calibrated likelihood, not a binary oracle. Keep a separate unknown/ambiguous state.
5. **Evaluation:** report payload recovery, false-positive rate, false-negative rate, perceptual quality, localization accuracy, runtime and memory over a fixed transformation matrix.
6. **Security:** rotate keys, rate-limit detector queries, test collusion and forgery, and assume a public detector can become an oracle for removal.
7. **Provenance:** sign the manifest, bind it to the original bytes, record the editing action, and use the signal watermark only as a recovery/matching path.
8. **Product language:** say “watermark detected” or “provenance claim validated” with the relevant confidence and scope. Do not say “human-made” merely because no signal was found.

A useful acceptance criterion is a measured operating point on the transformations that matter to the product. “Robust” without naming the codec, crop, resampling, paraphrase or detector threshold is not a reproducible claim.

## Sources

- [C2PA Specifications 2.4 — Content Credentials](https://spec.c2pa.org/specifications/specifications/2.4/specs/ContentCredentials.html)
- [C2PA Implementation Guidance — invisible watermarking and soft bindings](https://spec.c2pa.org/specifications/specifications/2.2/guidance/Guidance.html)
- [Google DeepMind — SynthID](https://deepmind.google/models/synthid/)
- [Google DeepMind — watermarking AI-generated text and video with SynthID](https://deepmind.google/blog/watermarking-ai-generated-text-and-video-with-synthid/)
- [Google DeepMind — identifying AI-generated images with SynthID](https://deepmind.google/blog/identifying-ai-generated-images-with-synthid/)
- [Fernandez et al. — VideoSeal: Open and Efficient Video Watermarking](https://arxiv.org/abs/2412.09492)
- [San Roman et al. — Proactive Detection of Voice Cloning with Localized Watermarking](https://arxiv.org/abs/2401.17264)
- [Fernandez et al. — AudioSeal implementation](https://github.com/facebookresearch/audioseal)
- [Fernandez et al. — VideoSeal implementation](https://github.com/facebookresearch/videoseal)
- [Fernandez et al. — Stable Signature](https://arxiv.org/abs/2303.15435)
- [Kirchenbauer et al. — A Watermark for Large Language Models](https://arxiv.org/abs/2301.10226)